# Étape 3 — Modélisation avec validation croisée

L’objectif de cette étape est de comparer plusieurs familles de modèles de classification pour prédire le risque de défaut crédit.

Le problème est une classification binaire :

- `TARGET = 0` : client sans défaut ;
- `TARGET = 1` : client en défaut.

Le dataset est fortement déséquilibré : la classe des clients en défaut représente environ 8 % des observations.
Il est donc nécessaire d’utiliser des métriques adaptées, comme le recall de la classe minoritaire, l’AUC-ROC, le F1-score et un coût métier personnalisé.

## Modèles comparés

Plusieurs familles de modèles sont testées :

| Famille | Modèle |
|---|---|
| Modèle linéaire | Logistic Regression |
| Forêt d’arbres | Random Forest |
| Boosting | LightGBM |
| Boosting | XGBoost |
| Réseau de neurones | MLPClassifier |

Les modèles sont évalués avec une validation croisée stratifiée afin de conserver la proportion de clients en défaut dans chaque fold.

## Validation croisée stratifiée

Un simple découpage train/validation peut donner une estimation instable des performances, car le résultat dépend du hasard du split.

Pour obtenir une estimation plus robuste, une validation croisée est utilisée avec `StratifiedKFold`.

La stratification permet de conserver la même proportion de clients en défaut dans chaque fold, ce qui est essentiel avec un dataset déséquilibré.

## Coût métier personnalisé

Dans ce projet, tous les types d’erreurs n’ont pas le même impact.

Un faux négatif correspond à un client réellement en défaut que le modèle classe comme non risqué.
C’est l’erreur la plus coûteuse, car l’entreprise risque d’accorder un crédit à un client qui ne remboursera pas.

Un faux positif correspond à un bon client classé comme risqué.
Cette erreur est moins grave, car l’entreprise refuse potentiellement un bon client, mais ne subit pas directement une perte de crédit.

Le coût métier utilisé est donc :

`business_cost = 10 × FN + 1 × FP`

In [1]:
from pathlib import Path

import mlflow
import pandas as pd


PROJECT_ROOT = Path.cwd().parent
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")

EXPERIMENT_NAME = "P6_credit_scoring_cross_validation"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(
        f"Expérience MLflow introuvable : {EXPERIMENT_NAME}. "
        f"Tracking URI utilisé : {mlflow.get_tracking_uri()}"
    )

print("Experiment ID :", experiment.experiment_id)
print("Tracking URI  :", mlflow.get_tracking_uri())

Experiment ID : 2
Tracking URI  : sqlite:////Users/vincentdesmouceaux/dev/P6_MLOps_credit_scoring/mlflow.db


In [2]:
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
)

columns_to_display = [
    "tags.mlflow.runName",
    "params.model_name",
    "params.n_splits",
    "params.sample_size",
    "params.n_features",
    "params.target_rate",
    "params.scale_pos_weight",
    "metrics.mean_accuracy",
    "metrics.std_accuracy",
    "metrics.mean_precision",
    "metrics.std_precision",
    "metrics.mean_recall",
    "metrics.std_recall",
    "metrics.mean_f1_score",
    "metrics.std_f1_score",
    "metrics.mean_roc_auc",
    "metrics.std_roc_auc",
    "metrics.mean_business_cost",
    "metrics.std_business_cost",
    "metrics.mean_business_cost_per_client",
    "metrics.std_business_cost_per_client",
]

available_columns = [
    column for column in columns_to_display
    if column in runs.columns
]

cv_results = runs[available_columns].copy()

cv_results

,tags.mlflow.runName,params.model_name,params.n_splits,params.sample_size,params.n_features,params.target_rate,params.scale_pos_weight,metrics.mean_accuracy,metrics.std_accuracy,metrics.mean_precision,...,metrics.mean_recall,metrics.std_recall,metrics.mean_f1_score,metrics.std_f1_score,metrics.mean_roc_auc,metrics.std_roc_auc,metrics.mean_business_cost,metrics.std_business_cost,metrics.mean_business_cost_per_client,metrics.std_business_cost_per_client
0,mlp_balanced_cv,mlp_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.918925,0.000297,0.317172,...,0.008051,0.011540,0.015400,0.021890,0.722402,0.013710,21380.000000,213.569661,0.801750,0.008016
1,xgboost_weighted_cv,xgboost_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.748125,0.003912,0.187761,...,0.637347,0.011037,0.290064,0.006169,0.768847,0.007945,13742.666667,290.775400,0.515350,0.010915
2,lightgbm_weighted_cv,lightgbm_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.799125,0.002709,0.210908,...,0.542892,0.011432,0.303784,0.005090,0.764717,0.006779,14212.666667,225.763888,0.532975,0.008472
3,random_forest_balanced_cv,random_forest_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.726925,0.007328,0.169173,...,0.608856,0.007797,0.264758,0.006041,0.736965,0.009731,14860.000000,257.423775,0.557250,0.009666
4,logistic_regression_balanced_cv,logistic_regression_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.706288,0.000895,0.169358,...,0.675750,0.010400,0.270836,0.002648,0.755332,0.004712,14114.333333,185.054947,0.529288,0.006942
5,mlp_balanced_cv,mlp_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.918925,0.000297,0.317172,...,0.008051,0.011540,0.015400,0.021890,0.722402,0.013710,21380.000000,213.569661,0.801750,0.008016
6,xgboost_weighted_cv,xgboost_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.748125,0.003912,0.187761,...,0.637347,0.011037,0.290064,0.006169,0.768847,0.007945,13742.666667,290.775400,0.515350,0.010915
7,lightgbm_weighted_cv,lightgbm_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.799125,0.002709,0.210908,...,0.542892,0.011432,0.303784,0.005090,0.764717,0.006779,14212.666667,225.763888,0.532975,0.008472
8,random_forest_balanced_cv,random_forest_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.726925,0.007328,0.169173,...,0.608856,0.007797,0.264758,0.006041,0.736965,0.009731,14860.000000,257.423775,0.557250,0.009666
9,logistic_regression_balanced_cv,logistic_regression_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.706288,0.000895,0.169358,...,0.675750,0.010400,0.270836,0.002648,0.755332,0.004712,14114.333333,185.054947,0.529288,0.006942


In [3]:
cv_results_sorted = cv_results.sort_values(
    by="metrics.mean_business_cost",
    ascending=True,
)

cv_results_sorted

,tags.mlflow.runName,params.model_name,params.n_splits,params.sample_size,params.n_features,params.target_rate,params.scale_pos_weight,metrics.mean_accuracy,metrics.std_accuracy,metrics.mean_precision,...,metrics.mean_recall,metrics.std_recall,metrics.mean_f1_score,metrics.std_f1_score,metrics.mean_roc_auc,metrics.std_roc_auc,metrics.mean_business_cost,metrics.std_business_cost,metrics.mean_business_cost_per_client,metrics.std_business_cost_per_client
1,xgboost_weighted_cv,xgboost_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.748125,0.003912,0.187761,...,0.637347,0.011037,0.290064,0.006169,0.768847,0.007945,13742.666667,290.775400,0.515350,0.010915
6,xgboost_weighted_cv,xgboost_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.748125,0.003912,0.187761,...,0.637347,0.011037,0.290064,0.006169,0.768847,0.007945,13742.666667,290.775400,0.515350,0.010915
10,xgboost_weighted_cv,xgboost_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.748125,0.003912,0.187761,...,0.637347,0.011037,0.290064,0.006169,0.768847,0.007945,13742.666667,290.775400,0.515350,0.010915
16,logistic_regression_balanced_cv,logistic_regression_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.706288,0.000895,0.169358,...,0.675750,0.010400,0.270836,0.002648,0.755332,0.004712,14114.333333,185.054947,0.529288,0.006942
4,logistic_regression_balanced_cv,logistic_regression_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.706288,0.000895,0.169358,...,0.675750,0.010400,0.270836,0.002648,0.755332,0.004712,14114.333333,185.054947,0.529288,0.006942
13,logistic_regression_balanced_cv,logistic_regression_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.706288,0.000895,0.169358,...,0.675750,0.010400,0.270836,0.002648,0.755332,0.004712,14114.333333,185.054947,0.529288,0.006942
9,logistic_regression_balanced_cv,logistic_regression_balanced_cv,3,80000,656,0.080725,11.387736141220191,0.706288,0.000895,0.169358,...,0.675750,0.010400,0.270836,0.002648,0.755332,0.004712,14114.333333,185.054947,0.529288,0.006942
2,lightgbm_weighted_cv,lightgbm_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.799125,0.002709,0.210908,...,0.542892,0.011432,0.303784,0.005090,0.764717,0.006779,14212.666667,225.763888,0.532975,0.008472
14,lightgbm_weighted_cv,lightgbm_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.799125,0.002709,0.210908,...,0.542892,0.011432,0.303784,0.005090,0.764717,0.006779,14212.666667,225.763888,0.532975,0.008472
7,lightgbm_weighted_cv,lightgbm_weighted_cv,3,80000,656,0.080725,11.387736141220191,0.799125,0.002709,0.210908,...,0.542892,0.011432,0.303784,0.005090,0.764717,0.006779,14212.666667,225.763888,0.532975,0.008472


In [4]:
summary_table = cv_results_sorted[
    [
        "tags.mlflow.runName",
        "metrics.mean_accuracy",
        "metrics.mean_precision",
        "metrics.mean_recall",
        "metrics.mean_f1_score",
        "metrics.mean_roc_auc",
        "metrics.mean_business_cost",
        "metrics.std_business_cost",
    ]
].copy()

summary_table = summary_table.rename(
    columns={
        "tags.mlflow.runName": "model",
        "metrics.mean_accuracy": "accuracy_mean",
        "metrics.mean_precision": "precision_mean",
        "metrics.mean_recall": "recall_mean",
        "metrics.mean_f1_score": "f1_mean",
        "metrics.mean_roc_auc": "roc_auc_mean",
        "metrics.mean_business_cost": "business_cost_mean",
        "metrics.std_business_cost": "business_cost_std",
    }
)

summary_table

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,business_cost_mean,business_cost_std
1,xgboost_weighted_cv,0.748125,0.187761,0.637347,0.290064,0.768847,13742.666667,290.775400
6,xgboost_weighted_cv,0.748125,0.187761,0.637347,0.290064,0.768847,13742.666667,290.775400
10,xgboost_weighted_cv,0.748125,0.187761,0.637347,0.290064,0.768847,13742.666667,290.775400
16,logistic_regression_balanced_cv,0.706288,0.169358,0.675750,0.270836,0.755332,14114.333333,185.054947
4,logistic_regression_balanced_cv,0.706288,0.169358,0.675750,0.270836,0.755332,14114.333333,185.054947
13,logistic_regression_balanced_cv,0.706288,0.169358,0.675750,0.270836,0.755332,14114.333333,185.054947
9,logistic_regression_balanced_cv,0.706288,0.169358,0.675750,0.270836,0.755332,14114.333333,185.054947
2,lightgbm_weighted_cv,0.799125,0.210908,0.542892,0.303784,0.764717,14212.666667,225.763888
14,lightgbm_weighted_cv,0.799125,0.210908,0.542892,0.303784,0.764717,14212.666667,225.763888
7,lightgbm_weighted_cv,0.799125,0.210908,0.542892,0.303784,0.764717,14212.666667,225.763888


## Résultats moyens de validation croisée

| Modèle | Accuracy moyenne | Precision moyenne | Recall moyen | F1 moyen | AUC moyenne | Coût métier moyen |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.706 | 0.169 | 0.676 | 0.271 | 0.755 | 14114 |
| Random Forest | 0.727 | 0.169 | 0.609 | 0.265 | 0.737 | 14860 |
| LightGBM | 0.799 | 0.211 | 0.543 | 0.304 | 0.765 | 14213 |
| XGBoost | 0.748 | 0.188 | 0.637 | 0.290 | 0.769 | 13743 |
| MLPClassifier | 0.919 | 0.317 | 0.008 | 0.015 | 0.722 | 21380 |

## Interprétation des résultats

La validation croisée montre que XGBoost obtient le meilleur coût métier moyen. Il présente également la meilleure AUC moyenne, ce qui indique une bonne capacité à distinguer les clients risqués des clients non risqués.

La Logistic Regression reste une baseline solide. Elle obtient le meilleur recall moyen, ce qui signifie qu’elle détecte une part importante des clients en défaut. Cependant, son coût métier moyen reste légèrement supérieur à celui de XGBoost.

LightGBM obtient une bonne accuracy, une bonne precision et un bon F1-score. En revanche, son recall est plus faible avec le seuil de décision actuel, ce qui pénalise son coût métier.

La Random Forest est moins performante dans cette première configuration.

Le MLPClassifier obtient une accuracy élevée, mais son recall est presque nul. Cela signifie qu’il détecte très peu de clients en défaut. Ce modèle est donc inadapté au contexte métier dans cette configuration.

## Conclusion

Cette étape a permis de comparer plusieurs familles de modèles avec une validation croisée stratifiée.

Les modèles simples, Logistic Regression et Random Forest, ont servi de premières références. Les modèles plus puissants, LightGBM et XGBoost, ont ensuite été comparés. Un MLPClassifier a également été testé afin d’évaluer une première approche de réseau de neurones.

Le modèle le plus performant selon le coût métier moyen est XGBoost. Il constitue donc le meilleur candidat actuel pour la suite du projet.

Cependant, les modèles sont encore évalués avec un seuil de décision par défaut. La prochaine étape consistera à optimiser ce seuil afin de minimiser davantage le coût métier.